# LoD3.2-reconstructie en gebouwverrijking

Deze notebook voert de volledige pijplijn uit voor het verrijken van een LoD2.2-gebouwmodel naar een LoD3-model met ramen, deuren, dakoversteken en balkons.

Alle herbruikbare logica staat in `functions_enrichment.py` en `visualize_cityjson.py`. De notebook bevat enkel de uitvoerbare stappen, parameters en controlevisualisaties.

In [ ]:
import functions_enrichment as fe
from pathlib import Path
import open3d as o3d
from PIL import Image, ImageDraw, ImageFont
import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import numpy as np
import importlib
import laspy
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
import json
import copy
import cityview as cv

## 1. Invoerdata en projectpaden

Stel hier de lokale projectmap, uitvoermap en databronnen in. De paden verwijzen naar de panoramabeelden, camerametadata, het LoD2.2-model, de gebouwmesh en de ALS-/MLS-puntenwolken.

In [ ]:
# Parameters
path = Path('/pad/naar/projectmap')
path_output = Path('/pad/naar/outputmap')

json_cam_path = path / 'camera parameters in .json formaat'
json_city_path = path / 'lod2.2-model in .json formaat'
mesh_building_path = path / 'lod2.2-model in .obj formaat'
ALS_path = path / 'ALS puntenwolk in .las of .laz formaat'
MLS_path = path / 'MLS puntenwolk in .las of .laz formaat'
panorama_folder = path / 'folder met 360°-Cycloramas'
panorama_ext = "afbeeldingsextentie van 360°-Cycloramas"

In [ ]:
# Uitvoering
# Data inladen
camera_data = fe.load_json(json_cam_path)
city_json = fe.load_json(json_city_path)
mesh = fe.load_mesh(mesh_building_path)
ALS = fe.load_lidar_data(ALS_path)
MLS = fe.load_lidar_data(MLS_path)

# Afbeeldingen verwerken
panorama_files = list(panorama_folder.glob(f"*{panorama_ext}"))
image_ids = [f.stem for f in panorama_files]

In [ ]:
# Controlevisualisatie van gebouwmesh en puntenwolk
o3d.visualization.draw_geometries([mesh, MLS])

## 2. Ramen en deuren

### 2.1 Eerste objectdetectie

Grounding DINO wordt gebruikt als open-set objectdetectiemodel. In deze stap worden de labels `window` en `door` op de originele panoramabeelden gedetecteerd.

De eerste detectie houdt voldoende kandidaatobjecten over. Deze detecties vormen de basis voor de eerste projectie naar 3D en de latere homografie.

In [ ]:
# Parameters
box_threshold1 = 0.3
text_threshold1 = 0.2

In [ ]:
# Modelconfiguratie
model_id = "IDEA-Research/grounding-dino-base"
#device = "cuda" if torch.cuda.is_available() else "cpu" # voor GPU
device = torch.device("mps") if torch.backends.mps.is_built() else torch.device("cpu") # voor mac met M4-pro chip

# Laad model en processor
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

In [ ]:
# Uitvoering
detections1_boxes = fe.detect_objects(panorama_folder, model, processor, device, panorama_ext, box_threshold1, text_threshold1, text="window. door.")

De detectieresultaten worden als controlebeelden opgeslagen, zodat de kwaliteit van de eerste bounding boxes visueel nagegaan kan worden.

In [ ]:
# Parameters
save_path_detections1 = path_output / '1_detections1_wd'

In [ ]:
# Controlebeelden opslaan
fe.save_detectionboxes_images(detections1_boxes, panorama_folder, save_path_detections1, panorama_ext)

### 2.2 Projectie naar 3D

#### Transformatiematrix

Per panorama wordt een cartesiaanse transformatiematrix opgesteld. Deze matrix koppelt pixelcoördinaten uit het panoramabeeld aan rays in het globale coördinatenstelsel.

In [ ]:
# Uitvoering
img_size_lookup = {img_id: Image.open(panorama_folder / f"{img_id}{panorama_ext}").size for img_id in image_ids}
panorama_parameters = fe.generate_cartesian_transforms(camera_data, img_size_lookup)

Controlevisualisatie van de camera-assen en beeldoriëntatie ten opzichte van mesh en puntenwolk.

In [ ]:
# Visualisatie
fe.visualize_camera_axes(panorama_parameters, mesh=mesh, point_cloud=ALS)

#### Raytracing

De hoekpunten van elke bounding box worden omgezet naar hoekpunten en vervolgens naar rays vanuit het cameracentrum.

In [ ]:
detections1_corners = fe.convert_boxes_to_corners(detections1_boxes)

De rays worden berekend op basis van de panoramaparameters en de hoekpunten van de eerste detecties.

In [ ]:
rays1 = fe.detections_to_rays(panorama_parameters, detections1_corners)

Controlevisualisatie van de rays ten opzichte van de gebouwmesh en puntenwolk.

In [ ]:
# Visualisatieparameter
ray_length = 20

In [ ]:
# Alle rays gelijktijdig visualiseren
fe.visualize_rays(rays1, mesh, ALS, ray_length)

#### Eerste 3D-snijpunten

In [ ]:
# Uitvoering
intersections1 = fe.calculate_intersections(mesh, rays1)

#### Filtering van 3D-detecties

Detecties zonder vier geldige snijpunten worden verwijderd.

In [ ]:
intersections_filt = fe.remove_zero_pointsets(intersections1)

Niet-coplanaire detecties worden verwijderd, omdat ramen en deuren als vlakke elementen worden gemodelleerd.

In [ ]:
intersections1 = fe.filter_coplanar_detections(intersections_filt)

In [ ]:
# Visualisatie van de intersectiepunten
fe.visualize_3d_points(intersections1,mesh)

De behouden 3D-detecties worden opnieuw op de oorspronkelijke panoramabeelden geprojecteerd als visuele controle.

In [ ]:
save_path_detections1_relevant = path_output / '2_detections1_relevant_wd'

fe.save_2d_points_with_labels(intersections1, panorama_folder, save_path_detections1_relevant, panorama_ext)

### 2.3 Tweede detectie op gerectificeerde beelden

#### Homografie

Op basis van de eerste 3D-snijpunten wordt per detectie een lokale homografie berekend. Hierdoor wordt het object frontaler voorgesteld voor een tweede, nauwkeurigere detectie.

In [ ]:
# Parameters
cropped_dir = path_output/'3_cropped_images_wd'
rectified_dir = path_output/'4_rectified_images_wd'

In [ ]:
# Uitvoering
transformation_parameters = fe.homographies_imageProcess(panorama_folder, intersections1,rectified_dir, cropped_dir=cropped_dir)

#### Objectdetectie op de gerectificeerde beelden

In [ ]:
# Parameters
box_threshold2 = 0.4
text_threshold2 = 0.2

In [ ]:
extension_rectified_image = '.jpg'
detections2_boxes = fe.detect_objects(rectified_dir, model, processor, device, extension_rectified_image, box_threshold2, text_threshold2, text="window. door.") 

In [ ]:
path_new_detections = path_output/'5_detections2_all_wd'
fe.save_detectionboxes_images(detections2_boxes, rectified_dir, path_new_detections, extension_rectified_image)

#### Selectie van relevante tweede detecties

Alleen bounding boxes die binnen de getransformeerde oorspronkelijke detectie vallen, worden behouden.

In [ ]:
detections2_boxes_relevant = fe.filter_bounding_boxes(transformation_parameters, detections2_boxes, offset=20)    #pixelcoord.

### 2.4 Tweede projectie naar 3D

#### Terugtransformatie naar panoramacoördinaten

De bounding boxes uit de gerectificeerde beelden worden met de inverse homografie teruggezet naar pixelcoördinaten in het oorspronkelijke panorama.

De homografie- en cropinformatie wordt toegevoegd aan de relevante detecties zodat de inverse transformatie reproduceerbaar blijft.

In [ ]:
for key, value in detections2_boxes_relevant.items():
    if key in transformation_parameters:
        # Voeg de H_offset en crop_offset toe aan elke entry in dict_2
        detections2_boxes_relevant[key] = {
            'bounding_boxes': value,
            'H_offset': transformation_parameters[key]['H_offset'],
            'crop_offset': transformation_parameters[key]['crop_offset'],
            'image_shape': transformation_parameters[key]['image_shape']
        }

De tweede detecties worden omgezet naar hoekpunten in het oorspronkelijke panoramabeeld.

In [ ]:
detections2_corners_pano_coordinates = fe.map_rectified_bboxes_to_original(detections2_boxes_relevant)  

In [ ]:
detections2_corners_pano_coordinates = fe.convert_annotations(detections2_corners_pano_coordinates)

Controlebeelden tonen welke tweede detecties effectief gebruikt worden voor de verdere verrijking.

In [ ]:
path_detections2_originalpict = path_output/ '6_detections2_relevant_wd'
fe.save_detectioncorners_images(detections2_corners_pano_coordinates, panorama_folder, path_detections2_originalpict, panorama_ext)

#### Raytracing

In [ ]:
rays2 = fe.detections_to_rays(panorama_parameters, detections2_corners_pano_coordinates)

In [ ]:
# Visualisatie
fe.visualize_rays(rays2, mesh, ALS, ray_length)

#### Verbeterde 3D-snijpunten

In [ ]:
intersections2 = fe.calculate_intersections(mesh, rays2)

Detecties zonder vier geldige snijpunten worden verwijderd.

In [ ]:
intersections2 = fe.remove_zero_pointsets(intersections2)

Niet-coplanaire resultaten worden verwijderd voordat de objecten gegroepeerd worden.

In [ ]:
intersections2 = fe.filter_coplanar_detections(intersections2)

Controlevisualisatie van de verbeterde 3D-snijpunten.

In [ ]:
fe.visualize_3d_points(intersections2, mesh)

De finale detecties worden terug op de originele beelden getekend als documentatie van de gebruikte objecten.

In [ ]:
converted_intersections2_save_image = fe.convert_intersections_to_detection_corners(intersections2)

In [ ]:
path_used_detections_for_enrichment = path_output/'7_used_detections_wd'
fe.save_detectioncorners_images(converted_intersections2_save_image, panorama_folder, path_used_detections_for_enrichment, panorama_ext)

### 2.5 Detecties groeperen en gemiddelde geometrie bepalen

Detecties van hetzelfde raam of dezelfde deur uit verschillende panoramabeelden worden gegroepeerd op basis van de afstand tussen overeenkomstige hoekpunten.

In [ ]:
grouped_wd = fe.group_similar_windows(intersections2, threshold=0.40)

for i, group in enumerate(grouped_wd):
    image_ids_in_group = [d['image_id'] for d in group]
    print(f"Groep {i+1}: {len(group)} detectie(s) uit beelden {image_ids_in_group}")
print(f"\nTotaal aantal unieke raam-/deurgroepen: {len(grouped_wd)}")

Per groep wordt een gemiddelde geometrie bepaald.

De iteratieve standaardafwijkingsfilter verwijdert uitschieters binnen elke groep.

De functie `mean_and_std_iteratief_filtered` berekent per groep het gemiddelde van de vier hoekpunten en behoudt in deze configuratie ook objecten die slechts één bruikbare detectie hebben.

In [ ]:
grouped_wd_mean = fe.mean_and_std_iteratief_filtered(grouped_wd, drempel=2.5, include_single_detections=True)

for entry in grouped_wd_mean:
    print(f"Raam/deur {entry['raamgroep']}: {entry['aantal_detecties']} detectie(s), "
          f"std_xy={entry['gemiddelde_standaardafwijking_xy_z'][0]:.3f}m, "
          f"std_z={entry['gemiddelde_standaardafwijking_xy_z'][1]:.3f}m")

De hoekfilter controleert of de overblijvende raam- en deurvlakken geometrisch plausibel zijn.

Alleen groepen met nagenoeg rechte hoeken worden behouden voor het wegschrijven in CityJSON.

In [ ]:
grouped_wd_filtered = fe.filter_raamgroepen_by_angle(grouped_wd_mean, tolerance_deg=3)
print(f"Raam-/deurgroepen voor filter: {len(grouped_wd_mean)}")
print(f"Raam-/deurgroepen na hoekfilter: {len(grouped_wd_filtered)}")

In [ ]:
# Omzetten naar visualisatieformaat
pts_final_converted_visualisation_wd = fe.convert_grouped_detections_to_visualization(grouped_wd_filtered)
print(f"Aantal finale raam-/deurelementen: {len(pts_final_converted_visualisation_wd)}")

In [ ]:
# Visualisatie
fe.visualize_3d_points(pts_final_converted_visualisation_wd, mesh=mesh)

### 2.6 Ramen en deuren wegschrijven in CityJSON

#### Coplanaire gevelvlakken samenvoegen

In [ ]:
city_json_coplanar = fe.merge_coplanar_adjacent_faces(city_json)

#### Vertexomrekening

CityJSON slaat vertices op als integercoördinaten met een schaal- en translatieparameter. De gedetecteerde wereldcoördinaten worden daarom naar CityJSON-vertices omgezet.

In [ ]:
pts_labels = fe.convert_visualization_to_points_labels(pts_final_converted_visualisation_wd)

In [ ]:
vertices_label_wd = fe.transform_to_vertices(pts_labels, city_json_coplanar)

#### Uitlijning met gevelvlakken

In [ ]:
normal_alligned_vertices_wd = fe.normal_allignment_to_faces(city_json_coplanar, vertices_label_wd, offset_distance=0)

#### Overlappende detecties verwijderen

In [ ]:
corrected_normal_alligned_vertices_wd = fe.filter_overlapping_planes(normal_alligned_vertices_wd)

#### Cut-outs en dagkanten toevoegen

Ramen en deuren worden als echte openingen in de gevelvlakken ingeschreven. Daarna wordt de LoD2.2-geometrie gekopieerd naar LoD3 en worden raam-/deurvlakken met dagkanten toegevoegd.

In [ ]:
# Cut-outs toevoegen aan LOD 2.2 (inner rings in muurvlakken)
city_json_coplanar_cutout = fe.add_cutouts_to_cityjson(city_json_coplanar, corrected_normal_alligned_vertices_wd)

# LOD 2.2 mét cutouts kopiëren naar LOD3 MultiSurface
updated = fe.duplicate_lod22_geometry(city_json_coplanar_cutout)

De verrijkte LoD3-geometrie krijgt de juiste boundaries, semantics en vertices en wordt tussentijds opgeslagen.

In [ ]:
LOD3_window = fe.add_windows_doors(updated, corrected_normal_alligned_vertices_wd, inset_distance=200)
# CityJSON opslaan
with open(path_output/"lod3_wd.json", "w") as f:
    json.dump(LOD3_window, f, indent=2)

## 3. Dakoversteken

### 3.1 Dakvlakken uitsplitsen en uitbreiden

Alle dakvlakken worden uit het verrijkte CityJSON-model gehaald op basis van hun semantiek.

In [ ]:
roof_surfaces = fe.extract_roof_surfaces(LOD3_window)

Per dakrand worden zoekstrips opgesteld. Die strips bepalen waar ALS-punten kunnen wijzen op een dakoversteek.

In [ ]:
roof_strips = fe.generate_roof_edge_strips(roof_surfaces, d_small=0.8, d_large=1.5)

### 3.2 Puntenwolkclassificatie

De ALS-puntenwolk wordt per dakrand geclassificeerd in een kleine zoekstrip en een grotere controlezone.

In [ ]:
las = laspy.read(ALS_path)
points = np.column_stack([las.x, las.y, las.z])
analysis = fe.classify_points_per_edge(roof_strips, points, max_plane_dist=0.30)

De classificatieresultaten worden gefilterd tot bevestigde dakoversteken op basis van puntdichtheid, RMSE, hoek, planariteit en randdekking.

In [ ]:
confirmed = fe.filter_confirmed_overhangs(
    analysis, roof_strips,
    points=points,                    # nodig voor tiebreak
    min_points=5,
    max_ratio_groot=0.25,
    rmse_tiebreak_threshold=0.005,    # wanneer her-segmenteren
    tight_plane_dist=0.05,            # met welke tolerantie
    max_angle_schuin=35.0,            # max hoek schuine punten vs dakvlak
    max_angle_horiz=10.0,             # max hoek horizontale punten vs verticaal
    max_rmse_schuin=0.10,             # max RMSE schuin vs dakvlak
    max_rmse_horiz=0.30,              # max RMSE horizontaal vs goothoogte
    min_coverage=0.30,                 # min dekkingsgraad punten langs rand (30%)
    max_plane_rmse=0.15                # max interne spreiding punten (planariteit)
)

In [ ]:
fe.print_overhang_summary(confirmed)

### 3.3 Wegschrijven van dakoversteken

Bevestigde dakoversteken worden als bijkomende `RoofSurface`-vlakken toegevoegd aan de LoD3-geometrie.

In [ ]:
overhang_strips = fe.generate_overhang_strips(confirmed)
LOD3_window_and_overhang = fe.add_overhangs_to_cityjson(LOD3_window, overhang_strips)

## 4. Balkons

### 4.1 Eerste objectdetectie

De balkonpijplijn hergebruikt de beeldgebaseerde detectie, raytracing en homografiestappen, maar wordt apart uitgevoerd om labelverwarring met ramen en deuren te vermijden.

De eerste detectie op de panoramabeelden houdt kandidaat-balkons over voor verdere projectie en rectificatie.

In [ ]:
# Parameters
box_threshold1 = 0.3
text_threshold1 = 0.2

In [ ]:
# Uitvoering
detections1_boxes = fe.detect_objects(panorama_folder, model, processor, device, panorama_ext, box_threshold1, text_threshold1, text="balcony.")

De eerste balkondetecties worden als controlebeelden opgeslagen.

In [ ]:
# Parameters
save_path_detections1 = path_output / '1_detections1_bk'

In [ ]:
# Controlebeelden opslaan
fe.save_detectionboxes_images(detections1_boxes, panorama_folder, save_path_detections1, panorama_ext)

### 4.2 Projectie naar 3D

#### Transformatiematrix

Dezelfde panoramaparameters worden gebruikt om de balkonbounding boxes naar rays in wereldcoördinaten om te zetten.

In [ ]:
# Uitvoering
img_size_lookup = {img_id: Image.open(panorama_folder / f"{img_id}{panorama_ext}").size for img_id in image_ids}
panorama_parameters = fe.generate_cartesian_transforms(camera_data, img_size_lookup)

Controlevisualisatie van camera-assen en beeldoriëntatie.

In [ ]:
# Visualisatie
fe.visualize_camera_axes(panorama_parameters, mesh=mesh, point_cloud=ALS)

#### Raytracing

De hoekpunten van de eerste balkonbounding boxes worden omgezet naar rays.

In [ ]:
detections1_corners = fe.convert_boxes_to_corners(detections1_boxes)

De rays worden aangemaakt op basis van de panoramaparameters.

In [ ]:
rays1 = fe.detections_to_rays(panorama_parameters, detections1_corners)

Controlevisualisatie van de balkonrays.

In [ ]:
# Visualisatieparameter
ray_length = 20

In [ ]:
# Alle rays gelijktijdig visualiseren
fe.visualize_rays(rays1, mesh, ALS, ray_length)

#### Eerste 3D-snijpunten

In [ ]:
# Uitvoering
intersections1 = fe.calculate_intersections(mesh, rays1)

#### Filtering van 3D-detecties

Detecties zonder vier geldige snijpunten worden verwijderd.

In [ ]:
intersections_filt = fe.remove_zero_pointsets(intersections1)

Niet-coplanaire detecties worden verwijderd.

In [ ]:
intersections1 = fe.filter_coplanar_detections(intersections_filt)

In [ ]:
# Visualisatie van de intersectiepunten
fe.visualize_3d_points(intersections1,mesh)

De behouden eerste balkondetecties worden opnieuw op de oorspronkelijke beelden geprojecteerd.

In [ ]:
save_path_detections1_relevant = path_output / '2_detections1_relevant_bk'

fe.save_2d_points_with_labels(intersections1, panorama_folder, save_path_detections1_relevant, panorama_ext)

### 4.3 Tweede detectie op gerectificeerde beelden

#### Homografie

De eerste balkonprojectie wordt gebruikt om een frontaler beeldfragment te maken waarop Grounding DINO opnieuw detecteert.

In [ ]:
# Parameters
cropped_dir = path_output/'3_cropped_images_bk'
rectified_dir = path_output/'4_rectified_images_bk'

In [ ]:
# Uitvoering
transformation_parameters = fe.homographies_imageProcess(panorama_folder, intersections1,rectified_dir, cropped_dir=cropped_dir)

#### Objectdetectie op de gerectificeerde beelden

In [ ]:
# Parameters
box_threshold2 = 0.4
text_threshold2 = 0.2

In [ ]:
extension_rectified_image = '.jpg'
detections2_boxes = fe.detect_objects(rectified_dir, model, processor, device, extension_rectified_image, box_threshold2, text_threshold2, text="balcony.") 

In [ ]:
path_new_detections = path_output/'5_detections2_all_bk'
fe.save_detectionboxes_images(detections2_boxes, rectified_dir, path_new_detections, extension_rectified_image)

#### Selectie van relevante tweede detecties

Alleen detecties binnen de oorspronkelijke getransformeerde balkonzone worden behouden.

In [ ]:
detections2_boxes_relevant = fe.filter_bounding_boxes(transformation_parameters, detections2_boxes, offset=20)    #pixelcoord.

### 4.4 Tweede projectie naar 3D en groepering

#### Terugtransformatie naar panoramacoördinaten

De tweede balkonbounding boxes worden teruggezet naar pixelcoördinaten in het oorspronkelijke panorama.

De transformatieparameters worden toegevoegd zodat de inverse homografie correct toegepast wordt.

In [ ]:
for key, value in detections2_boxes_relevant.items():
    if key in transformation_parameters:
        # Voeg de H_offset en crop_offset toe aan elke entry in dict_2
        detections2_boxes_relevant[key] = {
            'bounding_boxes': value,
            'H_offset': transformation_parameters[key]['H_offset'],
            'crop_offset': transformation_parameters[key]['crop_offset'],
            'image_shape': transformation_parameters[key]['image_shape']
        }

De tweede detecties worden omgezet naar hoekpunten in het oorspronkelijke panoramabeeld.

In [ ]:
detections2_corners_pano_coordinates = fe.map_rectified_bboxes_to_original(detections2_boxes_relevant)  

In [ ]:
detections2_corners_pano_coordinates = fe.convert_annotations(detections2_corners_pano_coordinates)

Controlebeelden tonen welke tweede balkondetecties gebruikt worden.

In [ ]:
path_detections2_originalpict = path_output/ '6_detections2_relevant_bk'
fe.save_detectioncorners_images(detections2_corners_pano_coordinates, panorama_folder, path_detections2_originalpict, panorama_ext)

#### Raytracing

In [ ]:
rays2 = fe.detections_to_rays(panorama_parameters, detections2_corners_pano_coordinates)

In [ ]:
# Visualisatie
fe.visualize_rays(rays2, mesh, ALS, ray_length)

#### Verbeterde 3D-snijpunten

In [ ]:
intersections2 = fe.calculate_intersections(mesh, rays2)

Detecties zonder vier geldige snijpunten worden verwijderd.

In [ ]:
intersections2 = fe.remove_zero_pointsets(intersections2)

Niet-coplanaire detecties worden verwijderd.

In [ ]:
intersections2 = fe.filter_coplanar_detections(intersections2)

Controlevisualisatie van de verbeterde balkonpunten.

In [ ]:
fe.visualize_3d_points(intersections2, mesh)

De finale balkondetecties worden op de originele beelden getekend als documentatie van de gebruikte objecten.

In [ ]:
converted_intersections2_save_image = fe.convert_intersections_to_detection_corners(intersections2)

In [ ]:
path_used_detections_for_enrichment = path_output/'7_used_detections_bk'
fe.save_detectioncorners_images(converted_intersections2_save_image, panorama_folder, path_used_detections_for_enrichment, panorama_ext)

#### Detecties uit meerdere beelden combineren

Detecties van hetzelfde balkon worden gegroepeerd. Voor balkons wordt een grotere afstandsdrempel gebruikt dan voor ramen en deuren, omdat de geprojecteerde gevelpositie sterker afwijkt door de uitkraging.

In [ ]:
grouped_bk = fe.group_similar_windows(intersections2, threshold=1.0) 

for i, group in enumerate(grouped_bk):
    image_ids_in_group = [d['image_id'] for d in group]
    print(f"Groep {i+1}: {len(group)} detectie(s) uit beelden {image_ids_in_group}")
print(f"\nTotaal aantal unieke balkons (groepen): {len(grouped_bk)}")

Per balkongroep wordt een gemiddelde geometrie berekend.

De iteratieve standaardafwijkingsfilter verwijdert uitschieters binnen elke balkongroep.

Groepen met één bruikbare detectie blijven behouden, omdat balkons niet altijd vanuit meerdere panoramastandpunten zichtbaar zijn.

In [ ]:
grouped_bk_mean = fe.mean_and_std_iteratief_filtered(grouped_bk, drempel=2.5, include_single_detections=True)

for entry in grouped_bk_mean:
    print(f"Balkon {entry['raamgroep']}: {entry['aantal_detecties']} detectie(s), "
          f"std_xy={entry['gemiddelde_standaardafwijking_xy_z'][0]:.3f}m, "
          f"std_z={entry['gemiddelde_standaardafwijking_xy_z'][1]:.3f}m")

De hoekfilter controleert of de gegroepeerde balkongeometrie voldoende rechthoekig is.

Alleen geometrisch plausibele balkongroepen worden verder gebruikt.

In [ ]:
grouped_bk_filtered = fe.filter_raamgroepen_by_angle(grouped_bk_mean, tolerance_deg=5)
print(f"Balkons voor filter: {len(grouped_bk_mean)}")
print(f"Balkons na hoekfilter: {len(grouped_bk_filtered)}")

In [ ]:
# Omzetten naar visualisatieformaat
pts_final_converted_visualisation_bk = fe.convert_grouped_detections_to_visualization(grouped_bk_filtered)
print(f"Aantal finale balkons: {len(pts_final_converted_visualisation_bk)}")

In [ ]:
# Visualisatie
fe.visualize_3d_points(pts_final_converted_visualisation_bk, mesh=mesh)

### 4.5 Balkonuitkraging en vloerplaat

#### Eerste reconstructie op basis van de detectie

Uit de onderste hoekpunten van het balkonvlak wordt een horizontale zoekstrip afgeleid. Deze strip bepaalt waar punten van de balkonvloer verwacht worden.

Per balkon wordt een strip gegenereerd aan de onderrand van de detectie. De strip steekt loodrecht uit de gevel en vormt de basis voor de puntenwolkmeting.

In [ ]:
overhang_strips = fe.generate_balkon_overhang_strips(grouped_bk_filtered, mesh, strip_breedte = 1.0,strip_lengte_marge = 0)
#overhang_strips_vertices = [strip['vertices'] for strip in overhang_strips]

#### Lokale puntenwolk rond de balkonvloer

De puntenwolk wordt lokaal uitgesneden zodat alleen punten rond het balkon overblijven.

In [ ]:
balkon_crops = fe.crop_balkon_points(overhang_strips, points)

#### Breedtebepaling van de balkonvloer

In [ ]:
overhang_cls = fe.classify_balkon_overhang_points(overhang_strips, balkon_crops, max_plane_dist_pass1=0.30,  max_plane_dist_pass2=0.10)
fe.print_balcony_summary(overhang_strips, overhang_cls)

### 4.6 Balkonhoogte via hertracering

#### Hertracering naar een virtueel vlak

In [ ]:
virtual_mesh = fe.create_virtual_plane_mesh(overhang_strips, overhang_cls, plane_height=8.0, plane_width_extra=2.0)

De gemeten uitkraging bepaalt een virtueel vlak voor de voorzijde van het balkon. De rays worden opnieuw gesneden met dit vlak om de bovenrand op de juiste diepte te bepalen.

In [ ]:
# Hertraceer rays naar virtueel vlak (zelfde functie, andere mesh)
intersections_virtual = fe.calculate_intersections(virtual_mesh, rays2)

In [ ]:
intersections_virtual = fe.remove_zero_pointsets(intersections_virtual)

In [ ]:
intersections_virtual = fe.filter_coplanar_detections(intersections_virtual)

In [ ]:
# Groepeer de hertraceerde detecties
grouped_virtual = fe.group_similar_windows(intersections_virtual, threshold=1.0)

for i, group in enumerate(grouped_virtual):
    image_ids_in_group = [d['image_id'] for d in group]
    print(f"Groep {i+1}: {len(group)} detectie(s) uit beelden {image_ids_in_group}")
print(f"\nTotaal aantal unieke balkons (groepen): {len(grouped_virtual)}")

In [ ]:
# Gemiddelde hoekpunten berekenen
grouped_virtual_mean = fe.mean_and_std_iteratief_filtered(
    grouped_virtual, 
    drempel=2.5, 
    include_single_detections=True
)

for entry in grouped_virtual_mean:
    print(f"Balkon {entry['raamgroep']}: {entry['aantal_detecties']} detectie(s), "
          f"std_xy={entry['gemiddelde_standaardafwijking_xy_z'][0]:.3f}m, "
          f"std_z={entry['gemiddelde_standaardafwijking_xy_z'][1]:.3f}m")

In [ ]:
# Hoekfilter
grouped_virtual_filtered = fe.filter_raamgroepen_by_angle(
    grouped_virtual_mean, 
    tolerance_deg=5
)

print(f"Balkons voor filter: {len(grouped_virtual_mean)}")
print(f"Balkons na hoekfilter: {len(grouped_virtual_filtered)}")

In [ ]:
# Visualisatie: gecorrigeerde balkonpunten op virtueel vlak
pts_virtual_vis = fe.convert_grouped_detections_to_visualization(grouped_virtual_filtered)
print(f"Aantal balkons op virtueel vlak: {len(pts_virtual_vis)}")
fe.visualize_3d_points(pts_virtual_vis, mesh=mesh)

### 4.7 Balkons toevoegen aan CityJSON

#### Geometrie opbouwen

De bovenste twee hoekpunten van de hertracering bepalen de bovenrand. De gemeten vloerplaat bepaalt de onderrand en uitkraging.

In [ ]:
# Combineer en exporteer
combined_balkons = fe.combine_virtual_and_overhang(grouped_virtual_filtered, overhang_strips, overhang_cls)

In [ ]:
balkons_compleet = fe.create_balkon_compleet(overhang_strips, overhang_cls, grouped_virtual_filtered)

#### Wegschrijven naar CityJSON

In [ ]:
LOD3_window_and_overhang_and_balkon = fe.add_balkons_to_cityjson(LOD3_window_and_overhang, balkons_compleet)

# CityJSON opslaan
with open(path_output/"lod3.json", "w") as f:
    json.dump(LOD3_window_and_overhang_and_balkon, f, indent=2)

## 5. Visualisatie en validatie van het eindmodel

In [ ]:
import visualize_cityjson as viz
import webbrowser

viz.generate_viewer(
    cityjson_data=LOD3_window_and_overhang_and_balkon,
    als_laz_path=ALS_path,
    output_path=path_output,
    mls_laz_path=MLS_path,       # bv: path / 'MLS_scan.laz'
    max_pts=150_000,
    pcd_radius=60.0,
)

# Viewer automatisch openen
viewer_path = path_output / 'viewer_lod3.html'
webbrowser.open(viewer_path.as_uri())
print(f"Viewer geopend: {viewer_path}")